In [115]:

import ifcopenshell as ifc
import ifcopenshell.util as ifc_util
import ifcopenshell.util.element as ifc_element
import ifcopenshell.util.placement as ifc_placement
import ifcopenshell.util.geolocation as ifc_geolocator
import glob 
import numpy as np
import pandas as pd
import ifcopenshell.geom
import ifcopenshell.util.shape
import random
import pathlib

from typing import Dict, Any, Optional, Literal

import os, sys, shutil

print("ifcopenshell version: ", ifc.__version__)

ifcopenshell version:  0.9.0.post1


In [89]:
path = glob.glob("BIM/*.ifc")
print(path[-1])
ifc_model = ifc.open(path[-1])
print(ifc_model.schema)
# for i in range(1,10):
#     random_entity = ifc_model.by_id(i)
#     print(random_entity.get_info())
    
# entity = ifc_model.by_type("IfcSIUnit")
# len(entity)
# walls = ifc_model.by_type("IfcWall")
# len(walls)
# print (walls[0].is_a("IfcWallStandardCase"))
# print (walls[0].is_a("IfcWall"))
# print (walls[0].id())
# print (walls[0].is_a("IfcWindow"))
# print (walls[0].is_a("IfcElement"))


BIM/edited.ifcview.ifc
IFC4


In [90]:
measuring_device = ifc_model.by_guid("17ivOSUPDAPhmNRfJI$rhR")
measuring_device.Tag = "Gadget"
measuring_device.get_info()
measuring_device.is_a("IfcElement")
measuring_device.get_info()

{'id': 11286,
 'type': 'IfcBuildingElementProxy',
 'GlobalId': '17ivOSUPDAPhmNRfJI$rhR',
 'OwnerHistory': None,
 'Name': 'X_ACCX_FT13_CEST04_0_Messpunkt',
 'Description': None,
 'ObjectType': None,
 'ObjectPlacement': #11296=IfcLocalPlacement(#91,#11295),
 'Representation': #11307=IfcProductDefinitionShape($,$,(#11306)),
 'Tag': 'Gadget',
 'PredefinedType': None}

In [91]:
# EXTRACT ALL MEASURING DEVICES WHICH STARTS WITH "G_"

gs = []
unique_measures = set()

for e in ifc_model.by_type("IfcElement"):
    try:
        name = e.Name
        if "_Messpunkt" in name:
            if name.startswith("G_"):
               e.Name = name.replace("_Messpunkt", "")
               gs.append(e)       
               unique_measures.add(e.Name.split("_")[1]) 
    except:
        pass

print("Number of measuring devices: ", len(gs))
print(unique_measures)

Number of measuring devices:  74
{'TEMP', 'NGST', 'LADS', 'ACCZ', 'PYRS', 'KKFT', 'IWAR', 'DMSK', 'NGSX', 'NGSY', 'DMSD', 'NGSF', 'NSSN', 'KKFF'}


In [92]:
all_elements = ifc_model.by_type("IfcElement")
len(all_elements)
ifc_element.get_psets(ifc_model.by_id(12860))


{'Sensorinformation': {'01 Produktbezeichnung': 'NA',
  '02 Status': 'eingebaut',
  '03 Kasten': 'Kasten Kistler',
  '04 Zusatz': 'NA',
  '05 Bemerkung': 'NA',
  '06 Laenge': 'NA',
  '07 Ansprechpartner': 'Kistler Instrumente',
  '08 DatumAnfang': '17.09.2025',
  '09 DatumEnde': 'NA',
  'id': 21515}}

In [93]:
df = pd.DataFrame(columns=["Name",
                           "QoI", 
                           "Units",
                           "sofistik_ref_number", 
                           "sofistik_grp_number", 
                           "IDP", "GlobalId", "Type", "AbsoluteCoordinates [X, Y, Z]", "RelativeCoordinates [X, Y, Z]", "RelativeToRef [X, Y, Z]"])

In [94]:
# UNITS {sensor_type: unit}
units = {
 'DMSD': {"unit": "μm/m", "QoI": "Strain"},
 'PYRS': {"unit": "W/m²", "QoI": "Solar Radiation"},
 'KKFT': {"unit": "°C", "QoI": "Temperature"},
 'KKFF': {"unit": "%", "QoI": "Humidity"},
 'IWAR': {"unit": "mm", "QoI": "Displacement"},
 'ACCZ': {"unit": "m/s²", "QoI": "Acceleration-Z"},
 'LADS': {"unit": "m", "QoI": "Distance"},
 'NGSF': {"unit": "%", "QoI": "Moisture"},
 'NGST': {"unit": "°C", "QoI": "Temperature"},
 'TEMP': {"unit": "°C", "QoI": "Temperature"},
 'DMSK': {"unit": "μm/m", "QoI": "Strain"},
 'NGSX': {"unit": "mm/m", "QoI": "Tilt-X"},
 'NSSN': {"unit": "unknown", "QoI": "unknown"},
 'NGSY': {"unit": "mm/m", "QoI": "Tilt-Y"},
 'ACCX': {"unit": "m/s²", "QoI": "Acceleration-X"},
 }

{'DMSD', 'PYRS', 'KKFT', 'KKFF', 'IWAR', 'ACCZ', 'LADS', 'NGSF', 'NGST', 'TEMP', 'DMSK', 'NGSX', 'NSSN', 'NGSY'}

# SMALL TRANSLATION DICTIONARY
# ... the english and german sensor labels differ   -\_**_/-

# ... de - en
# ... TF - PE
# ... BU - CB

small_dicts = {
    1: {"NGSX": "TILX", "NGSY": "TILY"}, 
    2: {"TF": "PE"},
    3: {"BU": "CB"}
}

In [95]:
settings = ifcopenshell.geom.settings()
settings.set("use-world-coords", True)



SPTs=[]
with open("OpenLAB_Data/Eisermann_2025_OPARA_FE-Models_openLAB/03_SOFiSTiK/openLAB_FE-Model.dat", "r") as f:
    for line in f:
        try:
            spt = line.split("SPT ")[1][:4]
            SPTs.append(int(spt))
        except:
            if "SPT " in line:
                print(f"ValueError: Could not convert SPT value to int for line:\n {line}")            
print(f"Forbidden SPTs: {SPTs}")

def gen_spt_number(element):
    # TODO: SPT rules
    # -- 1140 
    # -- ||||...... numb. ?
    # -- |||....... numb. ?
    # -- ||........ number of span
    # -- |......... number of axis
    name = element.Name
    parts = name.split("_")[2]
    if "FT" in parts:
        sptcandidate = int(str(parts.split("FT")[1]) + "00")
        niter_max = 100
        while sptcandidate in SPTs and niter_max > 0:
            sptcandidate += 1
            niter_max -= 1
        if niter_max == 0:
            raise ValueError("Maximum number of iterations reached, unable to safely generate the SPT number.")
        SPTs.append(sptcandidate)
        return sptcandidate
    else:
        return None
    
def gen_gpr_number(element):
    # TODO: GRP rules
    # -- 13
    # -- ||........ number of axis
    # -- |......... number of ???
    element_name = element.Name
    axis = element_name.split("_")[2]
    if "FT" in axis:
        axis_number = int(axis.split("FT")[1])
        gpr_number = 10 + axis_number
        return gpr_number
    else:
        return None
    
    
def get_element_absolute_coordinates(element):
    shape = ifcopenshell.geom.create_shape(settings, element)
    vertices = ifcopenshell.util.shape.get_vertices(shape.geometry)
    if len(vertices) > 0:
        coordinates = (vertices.min(axis=0) + vertices.max(axis=0)) / 2
        return np.round(coordinates, 3)
    else:
        return None

def get_element_relative_coordinates(element, full_coordinates=False):
    if element.ObjectPlacement:
        coordinates = ifc_placement.get_local_placement(element.ObjectPlacement)
        if full_coordinates:
            return np.round(coordinates, 3)
        else:
            return np.round(coordinates[:3,3], 3)
    else:
        return None


def gen_random_idp(existing_idps):
        return str(random.randint(1000, 9999))
        

IDPs = []
def gen_sofistik_idp(global_id):
    idp = str(global_id[:4])
    if idp not in IDPs:
        IDPs.append(idp)
    else:
        # gen random idp
        print(f"IDP {idp} already exists, generating a new one...")
        while True:
            idp = gen_random_idp(IDPs)
            if idp not in IDPs:
                IDPs.append(idp)
                break
    return idp

# Reference point 
# TODO: optimally use the #A10 REF_POINT, i.e. the end of the central axis, for the relative coordinates!
# REF_POINT = np.array([0., 0., 0.])

# Sensor with known position 
known_sensor_position = {"G_NGST_FT12_BU1100_0": np.array([11, -0.1, 0.])}
# shift of the reference coorditate system
for e in gs:
    if e.Name in list(known_sensor_position.keys()):
        abs_position_of_known_sensor = get_element_absolute_coordinates(e)
        REF_POINT = abs_position_of_known_sensor - known_sensor_position[e.Name]
        print(f"Shift vector for {e.Name}: {REF_POINT}")
        print(abs_position_of_known_sensor, known_sensor_position[e.Name])
        break


unique_yz_pairs = set()
unique_ys = set()
unique_zs = set()
def get_element_rel_to_ref_coordinates(relative_coordinates):
    coordinates = np.round(relative_coordinates - REF_POINT, 3)
    unique_yz_pairs.add((coordinates[1], coordinates[2]))
    unique_ys.add(coordinates[1])
    unique_zs.add(coordinates[2])
    return coordinates

for e in gs:
    # coordinates = get_element_coordinates(e, True)
    relative_coordinates = get_element_relative_coordinates(e, False)
    absolute_coordinates = get_element_absolute_coordinates(e)
    gpr_number = gen_gpr_number(e)
    reltoref_coorldinates = get_element_rel_to_ref_coordinates(absolute_coordinates)
    code = e.Name.split("_")[1]
    idp = gen_sofistik_idp(e.get_info().get("GlobalId", None))


    sofistik_ref_number = gen_spt_number(e)
    sofistik_grp_number = gen_gpr_number(e)

    df = pd.concat([df, pd.DataFrame([[
        e.get_info().get("Name", None),
        units.get(code, {}).get("QoI", None),
        units.get(code, {}).get("unit", None),
        sofistik_ref_number,
        sofistik_grp_number,  # TODO: GRP number
        idp,
        e.get_info().get("GlobalId", None), 
        e.is_a(), absolute_coordinates, relative_coordinates, reltoref_coorldinates,
        ]], columns=df.columns)], ignore_index=True)
    
df.to_csv("IFC_Sensor_Coordinates.csv", index=False)
print(f"Number of rows in the DataFrame: {len(df)}")
print(f"Unique Y values: {len(unique_ys)}")
print(f"Unique Z values: {len(unique_zs)}")
print(f"Unique YZ pairs: {len(unique_yz_pairs)}")
del unique_ys 
del unique_zs
df.head(10)

ValueError: Could not convert SPT value to int for line:
              SPT  #node+3000   REF PT #node   Z #H-#HOB+#dhspring

ValueError: Could not convert SPT value to int for line:
 SPT  #node+3000   REF PT #node   Z #H-#HOB+#dhspring

ValueError: Could not convert SPT value to int for line:
              SPT  #node+3005   REF PT #node   Z #H-#HOB+#dhspring XFLG A

ValueError: Could not convert SPT value to int for line:
              SPT  #node+3005   REF PT #node   Z #H-#HOB+#dhspring

Forbidden SPTs: [1100, 1110, 1140, 1170, 1190, 1200, 1210, 1220, 1230, 1240, 1250, 1260, 1280, 1281, 1290, 1291, 1300, 2100, 2110, 2140, 2170, 2190, 2200, 2210, 2220, 2230, 2240, 2250, 2260, 2280, 2281, 2290, 2291, 2300, 3100, 3110, 3140, 3170, 3190, 3200, 3210, 3220, 3230, 3240, 3250, 3260, 3280, 3281, 3290, 3291, 3300, 1101, 2101, 2102, 3101, 1301, 2301, 2302, 3301, 1105, 3105, 1201, 1202, 2201, 2202, 3201, 3202]
Shift vector for G_NGST_FT12_BU1100_0: [-32.68   0.63   2.11]
[-21.68   0.53   2.11] [1

,Name,QoI,Units,sofistik_ref_number,sofistik_grp_number,IDP,GlobalId,Type,"AbsoluteCoordinates [X, Y, Z]","RelativeCoordinates [X, Y, Z]","RelativeToRef [X, Y, Z]"
0,G_PYRS_UMFD_UW0000_0,Solar Radiation,W/m²,None,None,3S9S,3S9StnAXz56v_7$q2Xn0hr,IfcBuildingElementProxy,"[-32.6, 3.015, 3.39]","[-32.6, 2.965, 2.95]","[0.08, 2.385, 1.28]"
1,G_KKFF_UMFD_UW0000_0,Humidity,%,None,None,22BP,22BP_ITRz75BakPFOTTb0n,IfcBuildingElementProxy,"[-32.6, 3.015, 3.43]","[-32.6, 2.965, 2.95]","[0.08, 2.385, 1.32]"
2,G_KKFT_UMFD_UW0000_0,Temperature,°C,None,None,0kBn,0kBnalkLb5jhfHAa1$zTuB,IfcBuildingElementProxy,"[-32.6, 3.015, 3.47]","[-32.6, 2.965, 2.95]","[0.08, 2.385, 1.36]"
3,G_NSSN_UMFD_UW0000_0,unknown,unknown,None,None,1EoL,1EoLCjlnDDLOLkbEgv3KrZ,IfcBuildingElementProxy,"[-32.6, 3.015, 3.51]","[-32.6, 2.965, 2.95]","[0.08, 2.385, 1.4]"
4,G_NGSX_FT11_BU1100_0,Tilt-X,mm/m,1102,21,1hxd,1hxdFKOLnFu9cCbQG5d5Ll,IfcBuildingElementProxy,"[-21.72, 2.06, 2.07]","[-25.1, -0.85, 1.99]","[10.96, 1.43, -0.04]"
5,G_NGSY_FT11_BU1100_0,Tilt-Y,mm/m,1103,21,1fUZ,1fUZvilIDFneWjsOsqa6Mg,IfcBuildingElementProxy,"[-21.72, 2.06, 2.11]","[-25.1, -0.85, 1.99]","[10.96, 1.43, 0.0]"
6,G_NGST_FT11_BU1100_0,Temperature,°C,1104,21,2MLw,2MLwp1Kub28AHUyqkPbDPA,IfcBuildingElementProxy,"[-21.68, 2.06, 2.11]","[-25.1, -0.85, 1.99]","[11.0, 1.43, 0.0]"
7,G_NGSF_FT11_BU1100_0,Moisture,%,1106,21,3JMi,3JMiJK87r3Y96HIphmIlQX,IfcBuildingElementProxy,"[-21.68, 2.06, 2.07]","[-25.1, -0.85, 1.99]","[11.0, 1.43, -0.04]"
8,G_NGSX_FT12_BU1100_0,Tilt-X,mm/m,1203,22,0vjk,0vjkbwXFT7WOJhKq2CVlU5,IfcBuildingElementProxy,"[-21.72, 0.53, 2.07]","[-25.1, -0.85, 1.99]","[10.96, -0.1, -0.04]"
9,G_NGSY_FT12_BU1100_0,Tilt-Y,mm/m,1204,22,2g92,2g923mfA53uxcbWriaMTOq,IfcBuildingElementProxy,"[-21.72, 0.53, 2.11]","[-25.1, -0.85, 1.99]","[10.96, -0.1, 0.0]"


In [96]:
# Create SOFiSTIK input file for the sensors
try:
    os.remove("OpenLAB_sensors.dat")
    os.remove("OpenLAB_sensors_positions.dat")
except FileNotFoundError:
    pass
with open("OpenLAB_sensors.dat", "w") as f:
    f.write("!+prog sofimshc\n")
    f.write("!HEAD Sensor Definition\n")    
    f.write("!The IDP of sensors is a 4-digit code derived from the GlobalId of the IFC element (as the first 4 characters - if unique, otherwise generated randomly).\n")
    f.write("!Tilrmeters - the title always corresponds to the gadget label in GERMAN labeling (please... get rid of the redundant DE/EN labels... keep just one :D)\n")
    
    
    def write_sensortypes(df: pd.DataFrame, TYPE: Optional[Literal["S", "J", "A", "E", "H"]] = None, header: str = "", cadinp_lang: str = "DE") -> None:
        # df.sort_values(by="Name", key=lambda x: x.str.split("_")[2], inplace=True)
        df = df.copy()  # Create a copy of the DataFrame to avoid modifying the original
        def key_func(col):
            # Split the string by "_" and extract the third part
            third_part = col.str.split("_").str[2]
            # Extract the last two characters and convert to integer
            last_two_chars = third_part.str[-2:].str[::-1].astype(int)
            return last_two_chars

        df.sort_values(
            by="Name",
            key=key_func,
            inplace=True
        )
        
        if header:
            f.write(f"!{header}\n")
        for index, row in df.iterrows():
            # Get the position from the label
            name = row["Name"]
            span = name.split("_")[2][-2]
            girder = name.split("_")[2][-1]

            # TODO: CHECK WHETHER THIS HOLDS !!!
            reference_point = ["#A10", "#A20", "#A30"][int(span) - 1]  # Assuming span is 1, 2, or 3
            # SENSOR OFFSET FROM THE REFERENCE POINT (e.g., the girder's origin) - this is a placeholder, you need to compute it based on your specific requirements
            offset = float(name.split("_")[3][2:])/100  # Assuming the offset is the fourth part of the name



            position = reference_point + "+" + str(offset)  # commpute from the label
            spt_number = row["sofistik_ref_number"]
            grp_number = row["sofistik_grp_number"]

            comment = "! Sensor: " + row['Name'] + " - " + row['QoI'] + " - " + row['Units']
            match cadinp_lang:
                case "DE":
                    tp = "" if TYPE is None else f"TYP {TYPE} "
                    f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP {grp_number} SPT {spt_number} IDP '{row['IDP']}' BEZ '{row['Name']}' {comment}\n")
                    # f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP {grp_number} NQR ... SPT {spt_number} IDP '{row['IDP']}' BEZ '{row['Name']}' {comment}\n")
                case "EN":
                    tp = "" if TYPE is None else f"TYPE {TYPE} "
                    # f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP {grp_number} NCS ... SPT {spt_number} IDP '{row['IDP']}' TITL '{row['Name']}' {comment}\n")
                    f.write(f"GAXP A{girder} IDS 0 S {position}\t{tp} GRP {grp_number} SPT {spt_number} IDP '{row['IDP']}' TITL '{row['Name']}' {comment}\n")
                case _:
                    raise ValueError(f"Unsupported cadinp_lang: {cadinp_lang}, supported values are 'DE' and 'EN'. If you want to work with the openLAB model, use 'DE', because SOFiSTiK can not interpret 'EN' keywords when the original is in german.")
    
    write_sensortypes(df[df["QoI"] == "Tilt-X"], header="Tiltmeters - X")
    write_sensortypes(df[df["QoI"] == "Tilt-Y"], header="Tiltmeters - Y")

with open("OpenLAB_sensors_positions.dat", "w") as f:
    f.write("!Tilrmeters - positions\n")
    
    def write_sensortypes(df: pd.DataFrame, TYPE: Optional[Literal["S", "J", "A", "E", "H"]] = None, header: str = "", cadinp_lang: str = "DE") -> None:
        # df.sort_values(by="Name", key=lambda x: x.str.split("_")[2], inplace=True)
        df = df.copy()  # Create a copy of the DataFrame to avoid modifying the original
        def key_func(col):
            # Split the string by "_" and extract the third part
            third_part = col.str.split("_").str[2]
            # Extract the last two characters and convert to integer
            last_two_chars = third_part.str[-2:].str[::-1].astype(int)
            return last_two_chars

        df.sort_values(
            by="Name",
            key=key_func,
            inplace=True
        )
        
        if header:
            f.write(f"!{header}\n")
        for index, row in df.iterrows():
            # Get the position from the label
            name = row["Name"]
            span = name.split("_")[2][-2]
            girder = name.split("_")[2][-1]

            # TODO: CHECK WHETHER THIS HOLDS !!!
            reference_point = ["#A10", "#A20", "#A30"][int(span) - 1]  # Assuming span is 1, 2, or 3
            # SENSOR OFFSET FROM THE REFERENCE POINT (e.g., the girder's origin) - this is a placeholder, you need to compute it based on your specific requirements
            offset = float(name.split("_")[3][2:])/100  # Assuming the offset is the fourth part of the name



            position = reference_point + "+" + str(offset)  # commpute from the label
            spt_number = row["sofistik_ref_number"]
            grp_number = row["sofistik_grp_number"]

            comment = "! Sensor: " + row['Name'] + " - " + row['QoI'] + " - " + row['Units']
            match cadinp_lang:
                case "DE":
                    tp = "" if TYPE is None else f"TYP {TYPE} "
                    f.write(f"SPT {spt_number} X {position} Y {-row['RelativeToRef [X, Y, Z]'][1]}\n")
                case "EN":
                    tp = "" if TYPE is None else f"TYPE {TYPE} "
                    f.write(f"SPT {spt_number} X {position} Y {-row['RelativeToRef [X, Y, Z]'][1]}\n")
                case _:
                    raise ValueError(f"Unsupported cadinp_lang: {cadinp_lang}, supported values are 'DE' and 'EN'. If you want to work with the openLAB model, use 'DE', because SOFiSTiK can not interpret 'EN' keywords when the original is in german.")
    
    write_sensortypes(df[df["QoI"] == "Tilt-X"], header="Tiltmeters - X")
    write_sensortypes(df[df["QoI"] == "Tilt-Y"], header="Tiltmeters - Y")
    # write_sensortypes(df[df["QoI"] == "Tilt-X"][df[df["QoI"] == "Tilt-X"]["Name"].str.contains("_FT")], header="Tiltmeters - X")
    # write_sensortypes(df[df["QoI"] == "Tilt-Y"][df[df["QoI"] == "Tilt-Y"]["Name"].str.contains("_FT")], header="Tiltmeters - Y")
    # write_sensortypes(df[df["QoI"] == "Strain"], header="Strain gauges")
    # write_sensortypes(df[df["QoI"] == "Acceleration"], header="Acceleration sensors")

    # for index, row in df[df["QoI"] == "Tilt"].iterrows():
    #     f.write(f"GAXP {}")
    # for axis in ["A1", "A2", "A3"]:

    

    # ...
    f.write("!END")


In [97]:
df[df["QoI"] == "Tilt-X"].head(10)
# df[df["Name"] == "G_NGSX_FT13_BU1100_0"].head()

,Name,QoI,Units,sofistik_ref_number,sofistik_grp_number,IDP,GlobalId,Type,"AbsoluteCoordinates [X, Y, Z]","RelativeCoordinates [X, Y, Z]","RelativeToRef [X, Y, Z]"
4,G_NGSX_FT11_BU1100_0,Tilt-X,mm/m,1102,21,1hxd,1hxdFKOLnFu9cCbQG5d5Ll,IfcBuildingElementProxy,"[-21.72, 2.06, 2.07]","[-25.1, -0.85, 1.99]","[10.96, 1.43, -0.04]"
8,G_NGSX_FT12_BU1100_0,Tilt-X,mm/m,1203,22,0vjk,0vjkbwXFT7WOJhKq2CVlU5,IfcBuildingElementProxy,"[-21.72, 0.53, 2.07]","[-25.1, -0.85, 1.99]","[10.96, -0.1, -0.04]"
12,G_NGSX_FT13_BU1100_0,Tilt-X,mm/m,1302,23,11C2,11C21a$KX2_OpI09sZACZI,IfcBuildingElementProxy,"[-21.72, -0.98, 2.07]","[-25.1, -0.85, 1.99]","[10.96, -1.61, -0.04]"
16,G_NGSX_FT21_BU0400_0,Tilt-X,mm/m,2103,31,10xL,10xLGxmG97NflyiObcUx3U,IfcBuildingElementProxy,"[-13.52, 2.06, 2.07]","[-25.1, -0.85, 1.99]","[19.16, 1.43, -0.04]"
20,G_NGSX_FT22_BU0400_0,Tilt-X,mm/m,2203,32,0Qr2,0Qr206Efz1t9VIowmz6dey,IfcBuildingElementProxy,"[-13.52, 0.53, 2.07]","[-25.1, -0.85, 1.99]","[19.16, -0.1, -0.04]"
24,G_NGSX_FT23_BU0400_0,Tilt-X,mm/m,2303,33,04La,04LaetBFD4W9lVYrSLdmIC,IfcBuildingElementProxy,"[-13.52, -0.98, 2.07]","[-25.1, -0.85, 1.99]","[19.16, -1.61, -0.04]"


In [98]:
df[["Name", "QoI"]].to_csv("Sensor_Names.csv", index=False)

In [99]:
df_autoextract = pd.read_csv("Sensor_Names.csv")
# add column 
df_autoextract["sofistik_ref_number"] = None
df_autoextract.head(10)


sensors_of_interest = df_autoextract["Name"].tolist()

model_path = "OpenLAB_Data/Eisermann_2025_OPARA_FE-Models_openLAB/03_SOFiSTiK/openLAB_FE-Model-my.dat"
lines = []
with open(model_path, "r") as f:
    for line in f:
        if ("BEZ " in line) and ("SPT " in line):
            lines.append(line)
            
for sensor in sensors_of_interest:
    for line in lines:
        if sensor in line:                
            spt_number = int(line.split("SPT ")[1].split()[0])
            df_autoextract.loc[df_autoextract["Name"] == sensor, "sofistik_ref_number"] = spt_number
del lines

df_autoextract.to_csv("Sensor_Names_autoextracted_from_model.csv", index=False)
df_autoextract.head(10)

,Name,QoI,sofistik_ref_number
0,G_PYRS_UMFD_UW0000_0,Solar Radiation,None
1,G_KKFF_UMFD_UW0000_0,Humidity,None
2,G_KKFT_UMFD_UW0000_0,Temperature,None
3,G_NSSN_UMFD_UW0000_0,unknown,None
4,G_NGSX_FT11_BU1100_0,Tilt-X,1140
5,G_NGSY_FT11_BU1100_0,Tilt-Y,1140
6,G_NGST_FT11_BU1100_0,Temperature,None
7,G_NGSF_FT11_BU1100_0,Moisture,None
8,G_NGSX_FT12_BU1100_0,Tilt-X,2140
9,G_NGSY_FT12_BU1100_0,Tilt-Y,2140


In [ ]:
# prepare the extraction script
template = "OpenLAB_Data/Eisermann_2025_OPARA_FE-Models_openLAB/03_SOFiSTiK/openLAB_FE-Model-template.results"
filename = template.replace("-template", "")
shutil.copy(template, filename)

spt_of_interest = set(df_autoextract["sofistik_ref_number"].tolist())
spt_of_interest.discard(None)  # Remove None values if any
new_lines = []
for i,spt in enumerate(spt_of_interest):
    new_line = f'FLT  ID {i+1} NAME "n_disp.nr" RULE "{spt}"\n'
    new_lines.append(new_line)

path = pathlib.Path(filename)
path.write_text(path.read_text().replace("!__FLT_PLACEHOLDER__", "".join(new_lines)))
    
# print("New lines added to the extraction script:")
# print("".join(new_lines))
# print(spt_of_interest)

# with open(filename, "r") as f:
#     filedata = f.read()
# filedata = filedata.replace("!__FLT_PLACEHOLDER__", "".join(new_lines))
# with open(filename, "w") as f:
#     f.write(filedata)
    

1806